In [ ]:
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import VimeoVideo
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.utils.validation import check_is_fitted

warnings.simplefilter(action="ignore", category=FutureWarning)


In [ ]:
VimeoVideo("656752925", h="701f3f4081", width=600)


In [ ]:
def wrangle(filepath):
    # Read CSV file
    df = pd.read_csv(filepath)

    # Subset data: Apartments in "Capital Federal", less than 400,000
    mask_ba = df["place_with_parent_names"].str.contains("Capital Federal")
    mask_apt = df["property_type"] == "apartment"
    mask_price = df["price_aprox_usd"] < 400000
    df = df[mask_ba & mask_apt & mask_price]

    # Subset data: Remove outliers for "surface_covered_in_m2"
    low, high = df["surface_covered_in_m2"].quantile([0.1, 0.9])
    mask_area = df["surface_covered_in_m2"].between(low,high)
    df = df[mask_area]

    # Split lat-lon
    df[["lat","lon"]] = df["lat-lon"].str.split(",",expand=True).astype(float)
    df.drop(columns="lat-lon",inplace=True)
    
    return df


In [ ]:
VimeoVideo("656752771", h="3a42896eb6", width=600)


In [ ]:
frame1 = wrangle("data/buenos-aires-real-estate-1.csv")
print(frame1.info())
frame1.head()


In [ ]:
VimeoVideo("656751955", h="e47002428d", width=600)


In [ ]:
frame1
frame1.head()


In [ ]:
# Check your work
assert (
    frame1.shape[0] == 1343
), f"`frame1` should have 1343 rows, not {frame1.shape[0]}."
assert frame1.shape[1] == 17, f"`frame1` should have 17 columns, not {frame1.shape[1]}."


In [ ]:
VimeoVideo("656751853", h="da40b0a474", width=600)


In [ ]:
frame2 = wrangle("data/buenos-aires-real-estate-2.csv")
frame2.shape


In [ ]:
# Check your work
assert (
    frame2.shape[0] == 1315
), f"`frame1` should have 1315 rows, not {frame2.shape[0]}."
assert frame2.shape[1] == 17, f"`frame1` should have 17 columns, not {frame2.shape[1]}."


In [ ]:
VimeoVideo("656751405", h="d1f95ab108", width=600)


In [ ]:
df = pd.concat([frame1, frame2], ignore_index=True)
print(df.info())
df.head()


In [ ]:
# Check your work
assert df.shape == (2658, 17), f"`df` is the wrong size: {df.shape}"


In [ ]:
VimeoVideo("656751031", h="367be02e14", width=600)


In [ ]:
fig = px.scatter_mapbox(
    df,  # Our DataFrame
    lat= "lat",
    lon= "lon",
    width=600,  # Width of map
    height=600,  # Height of map
    color= "price_aprox_usd",
    hover_data=["price_aprox_usd"]  # Display price when hovering mouse over house
)

fig.update_layout(mapbox_style="open-street-map")

fig.show()


In [ ]:
VimeoVideo("656750669", h="574287f687", width=600)


In [ ]:
# Create 3D scatter plot
fig = px.scatter_3d(
    df,
    x= "lon",
    y= "lat",
    z= "price_aprox_usd",
    labels={"lon": "longitude", "lat": "latitude", "price_aprox_usd": "price"},
    width=600,
    height=500,
)

# Refine formatting
fig.update_traces(
    marker={"size": 4, "line": {"width": 2, "color": "DarkSlateGrey"}},
    selector={"mode": "markers"},
)

# Display figure
fig.show()


In [ ]:
VimeoVideo("656750457", h="09f5fe3962", width=600)


In [ ]:
features = ["lon","lat"]
X_train = df[features]
X_train.head()


In [ ]:
VimeoVideo("656750323", h="1a82090b9b", width=600)


In [ ]:
target = "price_aprox_usd"
y_train = df[target]
y_train.head()


In [ ]:
VimeoVideo("656750112", h="1ef669fe2b", width=600)


In [ ]:
y_mean = y_train.mean()
y_mean


In [ ]:
y_pred_baseline = [y_mean] * len(y_train)
y_pred_baseline[:5]


In [ ]:
VimeoVideo("656749994", h="50c71bf4e5", width=600)


In [ ]:
mae_baseline = mean_absolute_error(y_train, y_pred_baseline)

print("Mean apt price", round(y_mean, 2))
print("Baseline MAE:", round(mae_baseline, 2))


In [ ]:
VimeoVideo("656748776", h="014f943c46", width=600)


In [ ]:
imputer = SimpleImputer()


In [ ]:
# Check your work
assert isinstance(imputer, SimpleImputer)


In [ ]:
VimeoVideo("656748659", h="fdaa8d0329", width=600)


In [ ]:
imputer.fit(X_train)


In [ ]:
# Check your work
check_is_fitted(imputer)


In [ ]:
VimeoVideo("656748527", h="d76e63760c", width=600)


In [ ]:
XT_train = imputer.transform(X_train)
pd.DataFrame(XT_train, columns=X_train.columns).info()


In [ ]:
# Check your work
assert XT_train.shape == (2658, 2), f"`XT_train` is the wrong shape: {XT_train.shape}"
assert (
    np.isnan(XT_train).sum() == 0
), "Your feature matrix still has `NaN` values. Did you forget to transform is using `imputer`?"


In [ ]:
VimeoVideo("656748360", h="50b4643a26", width=600)


In [ ]:
model = make_pipeline(
    SimpleImputer(),
    LinearRegression()
)


In [ ]:
assert isinstance(model, Pipeline), "Did you instantiate your model?"


In [ ]:
VimeoVideo("656748234", h="59ba7958d5", width=600)


In [ ]:
model.fit(X_train, y_train)


In [ ]:
# Check your work
check_is_fitted(model["linearregression"])


In [ ]:
VimeoVideo("656748155", h="5672ef44cb", width=600)


In [ ]:
y_pred_training = model.predict(X_train)


In [ ]:
# Check your work
assert y_pred_training.shape == (2658,)


In [ ]:
VimeoVideo("656748205", h="13144556a6", width=600)


In [ ]:
mae_training = mean_absolute_error(y_train,y_pred_training)
print("Training MAE:", round(mae_training, 2))


In [ ]:
X_test = pd.read_csv("data/buenos-aires-test-features.csv")[features]
y_pred_test = pd.Series(model.predict(X_test))
y_pred_test.head()


In [ ]:
VimeoVideo("656747630", h="b90db6b373", width=600)


In [ ]:
intercept = model.named_steps["linearregression"].intercept_.round()
coefficients = model.named_steps["linearregression"].coef_.round()


In [ ]:
print(
    f"price = {intercept} + ({coefficients[0]} * longitude) + ({coefficients[1]} * latitude)"
)


In [ ]:
VimeoVideo("656746928", h="71bfe94764", width=600)


In [ ]:
# Create 3D scatter plot
fig = px.scatter_3d(
    df,
    x= "lon",
    y= "lat",
    z= "price_aprox_usd",
    labels={"lon": "longitude", "lat": "latitude", "price_aprox_usd": "price"},
    width=600,
    height=500,
)

# Create x and y coordinates for model representation
x_plane = np.linspace(df["lon"].min(), df["lon"].max(), 10)
y_plane = np.linspace(df["lat"].min(), df["lat"].max(), 10)
xx, yy = np.meshgrid(x_plane, y_plane)

# Use model to predict z coordinates
z_plane = model.predict(pd.DataFrame({"lon": x_plane, "lat": y_plane}))
zz = np.tile(z_plane, (10, 1))

# Add plane to figure
fig.add_trace(go.Surface(x=xx, y=yy, z=zz))

# Refine formatting
fig.update_traces(
    marker={"size": 4, "line": {"width": 2, "color": "DarkSlateGrey"}},
    selector={"mode": "markers"},
)

# Display figure
fig.show()
